# 第 5 周练习：动态 URL 的 RAG 系统

## 练习目标（理念）

用户输入任意网页 URL → 抓取页面（必要时跟链 / Selenium）→ 分块嵌入 → Chroma 向量库 → 检索增强问答（可开关：查询改写、重排、LLM 语义分块）。

## 和第 5 周概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 文档加载与分块 | `load_pages_from_url`、`RecursiveCharacterTextSplitter` / LLM chunking |
| 向量库与嵌入 | Chroma + `text-embedding-3-large`（经 OpenRouter） |
| RAG 检索增强 | `fetch_context` + `SYSTEM_PROMPT` 带 context |
| 进阶技巧 | 查询改写（rewrite）、LLM 重排（rerank）、结构化输出 |

## 怎么跑

1. `.env` 里配置 `OPENROUTER_API_KEY`
2. 从上到下运行；最后启动 Gradio，输入 URL 构建知识库再提问
3. 可选勾选 LLM 分块 / 改写 / 重排对比效果


In [ ]:
# ========== 导入：标准库 / 环境 / 抓取 / LangChain / 进阶工具 / UI ==========

# --- 标准库 ---
# 导入 os：读环境变量、检查向量库目录是否存在
import os
# 导入 re：正则处理（页面/文本清洗等）
import re
# 从 pathlib 导入 Path：跨平台路径对象
from pathlib import Path

# --- 环境 ---
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进进程环境
from dotenv import load_dotenv

# --- 网页解析（BeautifulSoup）---
# 从 bs4 导入 BeautifulSoup：解析 HTML、抽正文与标题
from bs4 import BeautifulSoup

# --- LangChain 生态 ---
# RecursiveUrlLoader：可递归跟链的页面加载器（本练习主路径也可能自写抓取）
from langchain_community.document_loaders import RecursiveUrlLoader
# RecursiveCharacterTextSplitter：按字符递归切块（经典分块基线）
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Document：LangChain 文档对象（page_content + metadata）
from langchain_core.documents import Document
# 消息类型：拼 RAG 对话（system / human / ai）
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
# OpenAIEmbeddings / ChatOpenAI：嵌入与聊天（可指向 OpenRouter base_url）
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# Chroma：持久化向量库封装
from langchain_chroma import Chroma

# --- 进阶：结构化输出、重试、并行、原生 OpenAI 客户端 ---
from openai import OpenAI
from chromadb import PersistentClient
from pydantic import BaseModel, Field
from tenacity import retry, wait_exponential
from tqdm import tqdm
from multiprocessing import Pool

# --- UI ---
import gradio as gr


In [ ]:
# ========== 环境：加载 .env 并检查 OpenRouter Key ==========

# override=True：.env 覆盖已有环境变量
load_dotenv(override=True)
# 读取 OpenRouter API Key（后面嵌入与聊天都走它）
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

# 仅打印前缀，确认 key 存在且不完整泄露
if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")


In [ ]:
# ========== 常量：向量库名、模型、分块与检索超参 ==========

# Chroma 持久化目录名
DB_NAME = "dynamic_vector_db"
# 嵌入模型 id（字符串保持原样）
EMBED_MODEL= "text-embedding-3-large"
# 对话 / 分块 / 重排用的聊天模型
LLM_MODEL = "gpt-4.1-nano"
# 经典分块：块大小与重叠（字符）
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
# 最终送给回答的 top-k
RETRIEVAL_K = 10
# 开启重排时先多取一些候选
RETRIEVAL_K_RERANK = 20
# LLM 分块时的进程池并行度
WORKERS = 3

# --- 全局状态：Gradio 会话间共享当前向量库与 URL ---
vectorstore = None
current_url = ""
vectorstore_state = ""


In [ ]:
# ========== OpenRouter 客户端 + 结构化输出辅助 ==========

# OpenAI 兼容网关
openrouter_url = "https://openrouter.ai/api/v1"

def get_client():
    # 每次拿一个指向 OpenRouter 的 OpenAI 客户端
    return OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)


def _pydantic_to_json_schema(pydantic_model: type[BaseModel]) -> dict:
    """把 Pydantic 模型转成 OpenAI response_format 所需的 json_schema 包装。"""
    schema = pydantic_model.model_json_schema()
    # OpenAI 需要 name + strict；schema 里可含 Pydantic 的 $defs
    return {
        "type": "json_schema",
        "json_schema": {
            "name": pydantic_model.__name__,
            "strict": True,
            "schema": schema,
        },
    }


def _completion(messages: list, model: str = None, json_mode: bool = False, response_format=None):
    """
    OpenAI 兼容 chat.completions。
    - json_mode：response_format={"type":"json_object"}
    - response_format：传入 Pydantic 类（如 ChunksSchema、RankOrder）做严格结构化输出
    """
    client = get_client()
    # 默认模型用全局 LLM_MODEL
    kwargs = {"model": model or LLM_MODEL, "messages": messages}
    if response_format is not None and isinstance(response_format, type) and issubclass(response_format, BaseModel):
        # 严格 JSON Schema 模式
        kwargs["response_format"] = _pydantic_to_json_schema(response_format)
    elif json_mode:
        # 宽松 JSON 对象模式
        kwargs["response_format"] = {"type": "json_object"}
    return client.chat.completions.create(**kwargs)


## 抓取（Scraping）

从 URL 拉 HTML：静态 `requests` 优先；若判定为 JS 渲染（SPA），可选 Selenium。可跟站内链接，最多抓若干页。


In [ ]:
# ========== 抓取：requests 静态页 +（可选）Selenium 渲染 + 跟链 ==========

import re
import time
import requests
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from langchain_core.documents import Document

# 可选：Selenium（需安装 selenium、webdriver-manager）；没有就退回纯静态抓取
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service
    from webdriver_manager.chrome import ChromeDriverManager
    SELENIUM_AVAILABLE = True
except ImportError:
    SELENIUM_AVAILABLE = False

# 浏览器 UA：降低被站点直接拒掉的概率
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

# JS 渲染启发式阈值：正文太短 + SPA 特征 → 考虑 Selenium
MIN_STATIC_CONTENT_CHARS = 300
SELENIUM_WAIT_SECONDS = 3


def _get_body_text(html: str) -> str:
    """只抽 body 纯文本，用于判断「是不是几乎空壳的 JS 页」。"""
    soup = BeautifulSoup(html, "html.parser")
    # 去掉脚本/样式/noscript，避免干扰正文长度统计
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    if soup.body:
        return soup.body.get_text(separator=" ", strip=True)
    return ""


def _has_spa_indicators(html: str) -> bool:
    """检查原始 HTML 是否出现常见 SPA/前端框架挂载点。"""
    lower = html.lower()
    patterns = [
        'id="root"',
        'id="app"',
        'id=\'root\'',
        'id=\'app\'',
        "__next_data__",
        "data-reactroot",
        "ng-version",
        "v-cloak",
        "vue-app",
        "react-root",
    ]
    return any(p in lower for p in patterns)


def is_js_rendered(html: str, url: str = "") -> bool:
    """启发式：正文很短且（有 SPA 特征或几乎空）→ 倾向于判定为 JS 渲染页。"""
    body_text = _get_body_text(html)
    text_len = len(body_text)

    if text_len >= MIN_STATIC_CONTENT_CHARS:
        return False

    if _has_spa_indicators(html):
        return True

    # 正文极少：也当作需要渲染/不可靠的静态壳
    if text_len < 100:
        return True

    return False


def extract_headers_and_content(html: str) -> tuple[list[str], str]:
    """返回 (标题列表, 正文纯文本)。标题形如 H1: ...；正文已去 script/style/nav 等。"""
    soup = BeautifulSoup(html, "html.parser")

    headers_list = []
    for tag in soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"]):
        text = tag.get_text(strip=True)
        if text:
            headers_list.append(f"{tag.name.upper()}: {text}")

    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    body = soup.get_text(separator="\n", strip=True)
    body = re.sub(r"\n\n+", "\n\n", body).strip()

    return headers_list, body


# 把 url+html 打成 LangChain Document（metadata 含 source/title）
def page_to_document(url: str, html: str, title: str = "") -> Document:
    """
    Create a LangChain Document with headers and content as separate fields.
    page_content combines both for embedding/retrieval.
    metadata holds source, title, headers, body.
    """
    headers_list, body = extract_headers_and_content(html)
    if not title:
        soup = BeautifulSoup(html, "html.parser")
        title = soup.title.string if soup.title and soup.title.string else "No title"

    page_content = "Headers:\n" + "\n".join(headers_list) + "\n\nContent:\n" + body

    return Document(
        page_content=page_content,
        metadata={
            "source": url,
            "title": title,
            "headers": "".join(headers_list),
            "body": body,
        },
    )


# 用 requests 拉静态 HTML
def fetch_html_requests(url: str) -> str:
    """Fetch raw HTML via requests (no JS execution)."""
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    return resp.text


# 用无头 Chrome 渲染后再取 page_source
def fetch_html_selenium(url: str) -> str:
    """Fetch rendered HTML via Selenium (JS executed)."""
    if not SELENIUM_AVAILABLE:
        raise RuntimeError("Selenium not installed. pip install selenium webdriver-manager")
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    try:
        driver.get(url)
        time.sleep(SELENIUM_WAIT_SECONDS)
        return driver.page_source
    finally:
        driver.quit()


# 加载单页：自动选择 requests 或 Selenium，并封装成 Document
def load_page(url: str, force_selenium: bool = False) -> Document:
    """
    Load a single page into a Document.
    - Tries requests first.
    - If is_js_rendered(html), falls back to Selenium.
    - Uses headers + content extraction and page_to_document.
    """
    if force_selenium:
        html = fetch_html_selenium(url)
    else:
        html = fetch_html_requests(url)
        if is_js_rendered(html, url):
            html = fetch_html_selenium(url)

    soup = BeautifulSoup(html, "html.parser")
    title = soup.title.string if soup.title and soup.title.string else ""

    return page_to_document(url, html, title)


# 入口：加载起始 URL，可选 follow_links 跟链直到 max_pages
def load_pages_from_url(
    url: str,
    follow_links: bool = False,
    max_pages: int = 5,
    force_selenium: bool = False,
) -> list[Document]:
    """
    Load the main page and optionally follow links.
    JS check and Selenium fallback apply to each page.
    """
    # 先抓起始页
    docs = [load_page(url, force_selenium=force_selenium)]

    # 不跟链则直接返回
    if not follow_links:
        return docs

    try:
        html = fetch_html_requests(url)
        if is_js_rendered(html, url) and SELENIUM_AVAILABLE:
            html = fetch_html_selenium(url)
    except Exception:
        return docs

    soup = BeautifulSoup(html, "html.parser")
    # seen：去重，避免环与重复抓取
    seen = {url}

    for a in soup.find_all("a", href=True):
        href = a["href"]
        full = urljoin(url, href) if not href.startswith("http") else href
        if not full.startswith("http") or full in seen:
            continue
        seen.add(full)
        if len(docs) >= max_pages:
            break
        try:
            docs.append(load_page(full, force_selenium=force_selenium))
        except Exception:
            pass

    return docs


## 分块（Chunking）

两条路径：**经典** `RecursiveCharacterTextSplitter`，或 **LLM 语义分块**（结构化输出 headline/summary/original_text）。问答质量很大程度取决于切块。


In [ ]:
# ========== 分块：Pydantic 结构 + LLM 语义切块 / 经典字符切块 ==========

# LLM 分块用的结构化字段：headline / summary / original_text

class ChunkSchema(BaseModel):
    # Field description 给模型看（英文保留）：影响结构化抽取质量
    headline: str = Field(
        description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query"
    )
    summary: str = Field(
        description="A few sentences summarizing the content of this chunk to answer common questions"
    )
    original_text: str = Field(
        description="The original text of this chunk from the provided document, exactly as is, not changed in any way"
    )


class ChunksSchema(BaseModel):
    chunks: list[ChunkSchema]


AVERAGE_CHUNK_SIZE = 200  # 提示模型大约按 ~200 字符一块来规划切分边界


def _make_llm_chunk_prompt(doc: Document) -> str:
    """构造「请把页面语义切块」的英文 prompt（影响行为，不翻译正文）。"""
    text = doc.page_content
    source = doc.metadata.get("source", "unknown")
    title = doc.metadata.get("title", "Unknown")
    # 粗估至少切成多少块，写进 prompt 给模型参考
    how_many = max(1, (len(text) // AVERAGE_CHUNK_SIZE) + 1)

    
    prompt = f"""
    You take a document from a web page and split it into overlapping chunks for a RAG knowledge base.

    The document is from: {source}
    Page title: {title}

    A chatbot will use these chunks to answer questions about the page content.
    You should divide the document as you see fit, ensuring the entire document is covered across the chunks - don't leave anything out.
    This document should probably be split into at least {how_many} chunks, but you can have more or less as appropriate.
    There should be overlap between chunks (typically ~25% or ~50 words) so the same context appears in multiple chunks for better retrieval.

    For each chunk provide:
    1. headline: a brief heading likely to match user queries
    2. summary: a few sentences summarizing the chunk for common questions
    3. original_text: the exact text of the chunk, unchanged

    Together your chunks must represent the entire document with overlap.

    Document:

    {text}

    Respond with the chunks in the required JSON format.
    """

    return prompt


@retry(wait=wait_exponential(multiplier=1, min=2, max=60))  # 失败指数退避重试
def _process_document_llm(doc: Document, model: str, client) -> list[Document]:
    """对单个 Document 做 LLM 分块，返回多个带 metadata 的 Document。"""
    
    messages = [{"role": "user", "content": _make_llm_chunk_prompt(doc)}]
    response = _completion(messages, model=model, response_format=ChunksSchema)
    reply = response.choices[0].message.content
    parsed = ChunksSchema.model_validate_json(reply)

    results = []
    base_metadata = {
        "source": doc.metadata.get("source", ""),
        "title": doc.metadata.get("title", ""),
        "type": "webpage",
    }
    for chunk in parsed.chunks:
        page_content = f"{chunk.headline}\n\n{chunk.summary}\n\n{chunk.original_text}"
        results.append(
            Document(page_content=page_content, metadata=base_metadata.copy())
        )
        
    return results


# 并行地对多文档做 LLM 分块（Pool + tqdm）
def create_chunks_llm(documents: list[Document], model: str = None, client=None) -> list[Document]:
    """
    Create chunks using LLM-based semantic splitting.
    Used when "Use LLM chunking" checkbox is enabled.
    """
    model = model or LLM_MODEL
    client = client or get_client()
    
    all_chunks = []
    with Pool(processes=WORKERS) as pool:
        for doc in tqdm(pool.imap_unordered(_process_document_llm, [documents, model, client]), desc="LLM chunking", total=len(documents)):
            # chunks = _process_document_llm(文档、模型、客户端)
            all_chunks.extend(doc)
    return all_chunks

def create_chunks(
    documents: list[Document],
    use_llm_chunking: bool = False,
) -> list[Document]:
    """统一入口：use_llm_chunking=True 走 LLM，否则走经典字符切块。"""
    if use_llm_chunking:
        return create_chunks_llm(documents)
    # 基线：按字符递归切分，带 overlap 保留跨块上下文
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    return splitter.split_documents(documents)


## 摄入（Ingest）

抓取 → 分块 → 嵌入 → 写入 Chroma。每次对新 URL 建库前会删掉旧 collection，避免新旧页面混在一起。


In [ ]:
# ========== 摄入：URL → chunks → Chroma（覆盖旧库）==========

def ingest_from_url(url: str, use_llm_chunking: bool = False) -> str:
    # 更新模块级向量库与当前 URL（Gradio 多回调共享）
    global vectorstore, current_url

    # 1) 抓取页面为 Document 列表
    docs = load_pages_from_url(url)
    
    if not docs:
        return f"Error: No content loaded from {url}"

    # 2) 分块（经典或 LLM）
    chunks = create_chunks(docs, use_llm_chunking)
    
    if not chunks:
        return f"Error: No chunks created from {url}"

    # 3) 嵌入模型（走 OpenRouter）
    embeddings = OpenAIEmbeddings(model=EMBED_MODEL, base_url=openrouter_url, api_key=openrouter_api_key)

    # 创建新库前清空已有 collection，避免混入上一次 URL 的向量
    if os.path.exists(DB_NAME):
        existing = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
        existing.delete_collection()

    # 4) 从 chunks 构建并持久化新的 Chroma
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_NAME,
    )
    # 新版 Chroma 持久化目录模式一般无需手动 persist()

    current_url = url
    return f"Indexed {len(chunks)} chunks from {url}"


## 检索（Retrieval）

可选：**查询改写**（多 query 合并）与 **LLM 重排**（先多取再按相关性排序，截断到 `RETRIEVAL_K`）。


In [ ]:
# ========== 检索：改写 query + 相似度搜索 +（可选）LLM 重排 ==========

# 重排结构化输出：order 是 1-based chunk id 列表（最相关在前）
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="Order of relevance (1-based chunk ids), most relevant first"
    )


def rewrite_query(question: str, history: list = None) -> str:
    """结合最近对话，把用户问题改写成更利于向量检索的短查询。"""
    history = history or []
    history_text = "\n".join(
        f"User: {m.get('content', m.get('message', ''))}"
        if isinstance(m, dict) else str(m)
        for m in history[-4:]
    )
    prompt = f"""
    You are about to search a knowledge base built from web pages.
    Create a short, focused query that will surface relevant content.
    Use only the rewritten query, nothing else.

    Conversation history:
    {history_text}

    Current question: {question}

    Rewrite as a concise search query:
    """
    response = _completion([{"role": "user", "content": prompt}])
    return response.choices[0].message.content.strip()


def rerank_chunks(question: str, chunks: list[Document]) -> list[Document]:
    """让 LLM 按与问题的相关性重排 chunks（结构化 order）。"""
    # 0/1 条无需重排
    if len(chunks) <= 1:
        return chunks
    system = """You are a document re-ranker. Order the chunks by relevance to the question.
Reply ONLY with a JSON object: {"order": [1, 2, 3, ...]} where numbers are 1-based chunk ids."""
    user = f"Question: {question}\n\nChunks:\n"
    for i, c in enumerate(chunks):
        user += f"# CHUNK ID {i+1}:\n{c.page_content[:500]}...\n\n"
    user += "Reply ONLY with JSON: {\"order\": [ ... ]}"
    response = _completion(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        response_format=RankOrder,
    )
    order = RankOrder.model_validate_json(response.choices[0].message.content).order
    return [chunks[i - 1] for i in order if 1 <= i <= len(chunks)]


# RAG 检索总入口：按开关决定是否改写 / 重排
def fetch_context(
    question: str,
    history: list = None,
    use_rewrite: bool = False,
    use_rerank: bool = False,
) -> list[Document]:
    """检索相关块。use_rewrite 合并原问+改写问；use_rerank 先多取再重排。"""
    # 尚未 ingest 时没有库
    if vectorstore is None:
        return []

    history = history or []
    # 重排开启时先取更多候选
    k = RETRIEVAL_K_RERANK if use_rerank else RETRIEVAL_K

    if use_rewrite:
        rewritten = rewrite_query(question, history)
        chunks1 = vectorstore.similarity_search(question, k=k)
        chunks2 = vectorstore.similarity_search(rewritten, k=k)
        # 改写查询的结果优先，再用原查询结果去重补齐
        seen_content = set()
        merged = []
        for c in chunks2:
            if c.page_content not in seen_content:
                seen_content.add(c.page_content)
                merged.append(c)
        for c in chunks1:
            if c.page_content not in seen_content:
                seen_content.add(c.page_content)
                merged.append(c)
        chunks = merged[:k]
    else:
        chunks = vectorstore.similarity_search(question, k=k)

    if use_rerank:
        chunks = rerank_chunks(question, chunks)

    return chunks[:RETRIEVAL_K]


## 回答问题（Answer）

把检索到的 context 填进 system prompt，用 `ChatOpenAI` 生成答案；另提供流式版本给 Gradio 打字机效果。


In [ ]:
# ========== 生成：system 带 context + 历史消息 +（流式）回答 ==========

# RAG system prompt：只根据 context 回答；英文模板保留（影响回答行为）
SYSTEM_PROMPT = """You are a helpful assistant. Answer the user's question using only the provided context from web pages they loaded.
If the context does not contain relevant information, say so. Do not make up facts.
When possible, cite the source (URL or page title).
Context:
{context}
"""

# LangChain ChatOpenAI，指向 OpenRouter；temperature=0 更稳
llm = ChatOpenAI(
    temperature=0,
    model=LLM_MODEL,
    base_url=openrouter_url,
    api_key=openrouter_api_key,
)


def _history_to_messages(history: list) -> list:
    """把 Gradio 历史转成 [{role, content}, ...]，供改写/检索使用。"""
    if not history:
        return []
    # 新版 messages 格式
    if isinstance(history[0], dict) and "role" in history[0]:
        return [{"role": m["role"], "content": m.get("content", "") or ""} for m in history]
    # 旧版 (user, bot) 元组格式
    msgs = []
    for user_msg, bot_msg in history:
        if user_msg:
            msgs.append({"role": "user", "content": user_msg})
        if bot_msg:
            msgs.append({"role": "assistant", "content": bot_msg})
    return msgs


def format_sources(chunks: list) -> str:
    """把检索到的块格式化成可点击来源 Markdown。"""
    if not chunks:
        return "_No sources retrieved._"
    lines = []
    for i, doc in enumerate(chunks, 1):
        source = doc.metadata.get("source", "Unknown")
        title = doc.metadata.get("title", "No title")
        preview = doc.page_content[:200].replace("\n", " ") + "..." if len(doc.page_content) > 200 else doc.page_content
        lines.append(f"**{i}. [{title}]({source})**\n   {preview}")
    return "\n\n".join(lines)


# 非流式回答：返回 (answer, sources_md, chunks)
def answer_question(
    question: str,
    history: list,
    use_rewrite: bool,
    use_rerank: bool,
) -> tuple[str, str, list]:
    """RAG 问答。返回 (答案, 来源 Markdown, 源 chunks)。"""
    if vectorstore is None:
        return "Please load a URL first to build the RAG system.", "_No sources._", []

    msg_history = _history_to_messages(history)
    docs = fetch_context(question, history=msg_history, use_rewrite=use_rewrite, use_rerank=use_rerank)

    if not docs:
        return "I couldn't find relevant context. Try rephrasing your question.", format_sources([]), []

    context = "\n\n---\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    messages = [SystemMessage(content=system_prompt)]
    for m in msg_history:
        if m["role"] == "user":
            messages.append(HumanMessage(content=m["content"]))
        else:
            messages.append(AIMessage(content=m["content"]))
    messages.append(HumanMessage(content=question))

    response = llm.invoke(messages)
    return response.content, format_sources(docs), docs


# 流式回答：逐 token yield (partial_answer, sources_md)
def answer_question_stream(
    question: str,
    history: list,
    use_rewrite: bool,
    use_rerank: bool,
):
    """
    Generator that streams the RAG answer token-by-token.
    Yields (partial_answer, sources_markdown).
    """
    if vectorstore is None:
        yield "Please load a URL first to build the RAG system.", "_No sources._"
        return

    msg_history = _history_to_messages(history)
    docs = fetch_context(question, history=msg_history, use_rewrite=use_rewrite, use_rerank=use_rerank)

    if not docs:
        yield "I couldn't find relevant context. Try rephrasing your question.", format_sources([])
        return

    context = "\n\n---\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    messages = [SystemMessage(content=system_prompt)]
    for m in msg_history:
        if m["role"] == "user":
            messages.append(HumanMessage(content=m["content"]))
        else:
            messages.append(AIMessage(content=m["content"]))
    messages.append(HumanMessage(content=question))

    sources_md = format_sources(docs)
    full_answer = ""
    # 流式：每来一块就累加并 yield，供 Gradio 刷新
    for chunk in llm.stream(messages):
        full_answer += chunk.content or ""
        yield full_answer, sources_md


## Gradio 用户界面

先输入 URL 构建 RAG，再聊天。可开关查询改写与重排；回答流式刷新，并展示来源片段。


In [ ]:
# ========== Gradio：构建库进度条 + 流式聊天 + 来源面板 ==========

custom_css = """
/* UI 主题：indigo/teal 渐变点缀（纯 CSS，不影响 RAG 逻辑） */
.gradio-container {
    font-family: 'Inter', 'SF Pro Display', system-ui, sans-serif !important;
    --primary-50: #eef2ff !important;
    --primary-100: #e0e7ff !important;
    --primary-500: #6366f1 !important;
    --primary-600: #4f46e5 !important;
    --primary-700: #4338ca !important;
    --teal-500: #14b8a6 !important;
    --teal-600: #0d9488 !important;
}
.header-box {
    text-align: center;
    padding: 2rem;
    background: linear-gradient(135deg, #4338ca 0%, #6366f1 50%, #06b6d4 100%);
    border-radius: 16px;
    color: white !important;
    font-weight: 600;
    box-shadow: 0 4px 20px rgba(99, 102, 241, 0.3);
}
.gr-button-primary {
    background: linear-gradient(135deg, #4f46e5, #06b6d4) !important;
    border: none !important;
    font-weight: 500 !important;
}
.gr-button-primary:hover {
    background: linear-gradient(135deg, #4338ca, #0d9488) !important;
    box-shadow: 0 4px 12px rgba(79, 70, 229, 0.4) !important;
}
.gr-input, .gr-textbox {
    border-radius: 10px !important;
    border: 1px solid #c7d2fe !important;
}
.gr-input:focus, .gr-textbox:focus-within {
    border-color: #6366f1 !important;
    box-shadow: 0 0 0 2px rgba(99, 102, 241, 0.2) !important;
}
.gr-form {
    border-radius: 12px !important;
}
"""


def build_rag_with_progress(url: str, use_llm_chunking: bool, progress=gr.Progress()) -> tuple[str, bool]:
    """带进度条地从 URL ingest；返回 (状态文案, 是否成功)。"""
    if not url or not url.strip().startswith("http"):
        return "Please enter a valid URL (e.g. https://en.wikipedia.org/wiki/...)", False
    try:
        progress(0.2, desc="Building RAG pipeline...")
        result = ingest_from_url(url.strip(), use_llm_chunking)
        progress(1.0, desc="Done!")
        return result, True
    except Exception as e:
        return f"Error: {str(e)}", False


def chat_fn(message, history, use_rewrite, use_rerank):
    """聊天回调：流式更新 history 最后一条 assistant，并附带来源 Markdown。"""
    if not message or not message.strip():
        return history, ""
    msg = message.strip()
    # Gradio messages 格式：[{role, content}, ...]
    new_history = history + [
        {"role": "user", "content": msg},
        {"role": "assistant", "content": ""}
    ]
    sources_md = ""
    for partial, sources_md in answer_question_stream(msg, history, use_rewrite, use_rerank):
        new_history[-1]["content"] = partial
        yield new_history, sources_md


with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue="indigo")) as demo:
    gr.Markdown("# Dynamic URL RAG System", elem_classes=["header-box"])

    rag_ready = gr.State(value=False)

    # === 顶部 URL 栏（RAG 就绪时可见）===
    with gr.Row(visible=False) as top_url_row:
        with gr.Column(scale=4):
            url_input_top = gr.Textbox(
                label="Load new URL",
                placeholder="https://en.wikipedia.org/wiki/...",
                show_label=True,
            )
        # 与 gr.Column(scale=1):
            # use_llm_chunk_top = gr.Checkbox(label="LLM 分块", value=False)
        with gr.Column(scale=1):
            build_btn_top = gr.Button("Build RAG", variant="primary")

    # === 初始居中 URL 输入（RAG 未就绪时可见）===
    with gr.Row(visible=True) as initial_url_row:
        with gr.Column(scale=1, min_width=200):
            pass
        with gr.Column(scale=3, min_width=400):
            url_input = gr.Textbox(
                label="Enter a URL to build your RAG knowledge base",
                placeholder="https://en.wikipedia.org/wiki/Python_(programming_language)",
                lines=1,
                show_label=True,
            )
            with gr.Row():
                # use_llm_chunk = gr.Checkbox(label="使用 LLM 分块", value=False)
                build_btn = gr.Button("Build RAG System", variant="primary", scale=2)
        with gr.Column(scale=1, min_width=200):
            pass

    status_text = gr.Markdown("", elem_id="status")

    # === 聊天部分（在 RAG 准备好之前隐藏）；状态也出现在这里 ===
    with gr.Row(visible=False) as chat_row:
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Chat", height=500, type="messages")
            with gr.Row():
                use_rewrite = gr.Checkbox(label="Use query rewriting", value=False)
                use_rerank = gr.Checkbox(label="Use reranking", value=False)
            msg_input = gr.Textbox(
                placeholder="Ask a question about the loaded page...",
                show_label=False,
                container=False,
            )
            msg_btn = gr.Button("Send", variant="primary")
        with gr.Column(scale=1):
            gr.Markdown("### Chunk references & sources")
            sources_output = gr.Markdown("_Sources will appear here after you ask a question._")

    # --- 构建 RAG（初始按钮）---
    def on_build(url, progress=gr.Progress()):
      # 第一个产量：隐藏聊天和来源，显示加载文本
        yield (
            gr.update(visible=True),   # initial_url_row stays
            gr.update(visible=False),  # chat_row — HIDE
            gr.update(visible=False),  # top_url_row
            gr.update(value=False),
            gr.update(value="**Building RAG pipeline...** Please wait."),
        )
        status, ok = build_rag_with_progress(url, False, progress)  # pass use_llm_chunking=False
        yield (
            gr.update(visible=not ok),
            gr.update(visible=ok),
            gr.update(visible=ok),
            gr.update(value=ok),
            gr.update(value=status),
        )

    build_btn.click(
        fn=on_build,
        inputs=[url_input],
        outputs=[initial_url_row, chat_row, top_url_row, rag_ready, status_text],
    )

    # --- 构建 RAG（顶部栏，加载新 URL 时）---
    def on_build_top(url, progress=gr.Progress()):
      # 第一个收益：隐藏聊天和来源
        yield (
            gr.update(visible=True),
            gr.update(visible=False),  # chat_row — HIDE
            gr.update(visible=False),
            gr.update(value=False),
            gr.update(value="**Building RAG pipeline...** Please wait."),
        )
        status, ok = build_rag_with_progress(url, False, progress)
        yield (
            gr.update(visible=not ok),
            gr.update(visible=ok),
            gr.update(visible=ok),
            gr.update(value=ok),
            gr.update(value=status),
        )


    # 顶部「构建」按钮：ingest 成功后切换到聊天布局
    build_btn_top.click(
        fn=on_build_top,
        inputs=[url_input_top],
        outputs=[initial_url_row, chat_row, top_url_row, rag_ready, status_text],
    )

    # --- 聊天发送（按钮 / Enter）---
    def respond(message, history, rw, rk):
        yield from chat_fn(message, history, rw, rk)

    msg_btn.click(
        fn=respond,
        inputs=[msg_input, chatbot, use_rewrite, use_rerank],
        outputs=[chatbot, sources_output],
    ).then(lambda: "", None, msg_input)

    msg_input.submit(
        fn=respond,
        inputs=[msg_input, chatbot, use_rewrite, use_rerank],
        outputs=[chatbot, sources_output],
    ).then(lambda: "", None, msg_input)

# queue：让流式 yield 能正确推到前端
demo.queue()
demo.launch()
